# 01 — Download NDVI from GEE

Export per-tile Sentinel-2 NDVI for each year (2016–2025) to Google Drive.

**Why the asset step?**  
GEE embeds the full FeatureCollection geometry into every task request. 33k inline
features exceed the 10 MB limit regardless of geometry type. Uploading tiles as a GEE
asset stores the geometry server-side; the task request then contains only a string
reference (~100 bytes).

**Run order:**
1. Cell 3 — saves `tiles_100m2_centroids.csv` and prints the upload command
2. Run that command in a terminal (one-time, ~1 min upload)
3. Cell 4 — waits until asset is ready
4. Cell 5 — loads `hex_fc` from asset
5. Cells 6–7 — define helpers and run the export loop

In [1]:
import ee
import geopandas as gpd
import pandas as pd
import json, os

TILES_PKL = '../data/polygons/tiles_100m2.pkl'
OUT_DIR   = '../data/ndvi/tiles_100m2'
os.makedirs(OUT_DIR, exist_ok=True)


/home/simonhans/anaconda3/lib/python3.7/site-packages/google/auth/crypt/_cryptography_rsa.py:22: CryptographyDeprecationWarning: Python 3.7 is no longer supported by the Python core team and support for it is deprecated in cryptography. The next release of cryptography will remove support for Python 3.7.
  import cryptography.exceptions
/home/simonhans/anaconda3/lib/python3.7/site-packages/geopandas/_compat.py:115: UserWarning: The Shapely GEOS version (3.11.4-CAPI-1.17.4) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  shapely_geos_version, geos_capi_version_string


In [2]:
# Authenticate once if needed
ee.Authenticate()
ee.Initialize(project='ee-simonhansedasi')


In [3]:
tiles = pd.read_pickle(TILES_PKL)
print(f'Tiles: {len(tiles)}')

# Save centroids as CSV — earthengine CLI uploads points via lon/lat columns
cents = tiles.copy()
cents['lon'] = cents.geometry.centroid.x
cents['lat']  = cents.geometry.centroid.y
CENTS_CSV = '../data/polygons/tiles_100m2_centroids.csv'
cents[['tile_id', 'lon', 'lat']].to_csv(CENTS_CSV, index=False)

ASSET_ID = 'projects/ee-simonhansedasi/assets/tiles_100m2_pts'

print(f'\nSaved: {CENTS_CSV}')
print('\nRun this in a terminal (one-time upload, ~1 min):')
print(f'  earthengine upload table \\')
print(f'    --asset_id={ASSET_ID} \\')
print(f'    --x_column=lon --y_column=lat \\')
print(f'    {CENTS_CSV}')
print('\nThen run the next cell to wait for the asset to be ready.')

Tiles: 32978


/home/simonhans/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  
/home/simonhans/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  import sys



Saved: ../data/polygons/tiles_100m2_centroids.csv

Run this in a terminal (one-time upload, ~1 min):
  earthengine upload table \
    --asset_id=projects/ee-simonhansedasi/assets/tiles_100m2_pts \
    --x_column=lon --y_column=lat \
    ../data/polygons/tiles_100m2_centroids.csv

Then run the next cell to wait for the asset to be ready.


In [4]:
import time

# Poll until the asset ingestion task completes (usually < 2 min)
print('Waiting for asset to be ready...')
while True:
    tasks = ee.data.listOperations(f'projects/ee-simonhansedasi')
    ingestion = [t for t in tasks if 'tiles_100m2_pts' in t.get('metadata', {}).get('description', '')]
    if ingestion:
        state = ingestion[0].get('metadata', {}).get('state', 'UNKNOWN')
        print(f'  Ingestion state: {state}')
        if state in ('SUCCEEDED', 'COMPLETED'):
            break
        if state in ('FAILED', 'CANCELLED'):
            print('Ingestion failed — check the earthengine upload command.')
            break
    else:
        # Asset may already exist or task not visible yet
        try:
            ee.data.getAsset(ASSET_ID)
            print('Asset found.')
            break
        except Exception:
            pass
    time.sleep(15)

Waiting for asset to be ready...
Asset found.


In [5]:
# The CLI upload stores lon/lat as properties but creates empty geometry.
# Reconstruct Point geometry server-side from those properties — no re-upload needed.
hex_fc = ee.FeatureCollection(ASSET_ID).map(
    lambda f: f.setGeometry(ee.Geometry.Point([f.getNumber('lon'), f.getNumber('lat')]))
)
print(f'hex_fc: {hex_fc.size().getInfo()} features')
print('First geometry:', hex_fc.first().geometry().getInfo())

hex_fc: 32978 features
First geometry: {'type': 'Point', 'coordinates': [-119.7820736613326, 45.87247052903091]}


In [6]:
def add_ndvi(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('ndvi')
    return image.addBands(ndvi)

def extract_image(img):
    # sampleRegions samples the pixel containing each Point — no buffer needed
    return (img.select('ndvi')
        .sampleRegions(
            collection=hex_fc,
            scale=10,
            geometries=False,
            tileScale=4,
        )
        .map(lambda f: f.set('date', img.date().format('YYYY-MM-dd')))
    )

In [7]:
# ── Validate geometry fix, then test sampling ────────────────────────────────

import pandas as pd
cents = pd.read_csv('../data/polygons/tiles_100m2_centroids.csv')
lon_min, lon_max = cents['lon'].min(), cents['lon'].max()
lat_min, lat_max = cents['lat'].min(), cents['lat'].max()
print(f'Centroid bbox: lon [{lon_min:.5f}, {lon_max:.5f}]  lat [{lat_min:.5f}, {lat_max:.5f}]')

aoi = ee.Geometry.BBox(lon_min, lat_min, lon_max, lat_max)

test_col = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate('2022-07-01', '2022-08-01')
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .select(['B4', 'B8'])
      .map(add_ndvi)
)
print('Images in July 2022:', test_col.size().getInfo())

# Sample 5 tiles — should now return ndvi values
sample = test_col.first().select('ndvi').sampleRegions(
    collection=hex_fc.limit(5),
    scale=10,
    geometries=False,
)
print('Sample (5 tiles):', sample.getInfo())
assert sample.size().getInfo() > 0, "Still returning 0 features — geometry fix didn't work"
print('Geometry fix confirmed — proceeding to export loop.')

Centroid bbox: lon [-119.79142, -119.73180]  lat [45.86670, 45.89293]
Images in July 2022: 9
Sample (5 tiles): {'type': 'FeatureCollection', 'columns': {}, 'properties': {'band_order': ['ndvi']}, 'features': [{'type': 'Feature', 'geometry': None, 'id': '00000000000000000000_0', 'properties': {'lat': 45.87247052903091, 'lon': -119.7820736613326, 'ndvi': 0.3565300405025482, 'tile_id': 0}}, {'type': 'Feature', 'geometry': None, 'id': '00000000000000000001_0', 'properties': {'lat': 45.8724678030942, 'lon': -119.7819642116172, 'ndvi': 0.35477766394615173, 'tile_id': 1}}, {'type': 'Feature', 'geometry': None, 'id': '00000000000000000002_0', 'properties': {'lat': 45.87246814625692, 'lon': -119.78184425494813, 'ndvi': 0.2955082654953003, 'tile_id': 2}}, {'type': 'Feature', 'geometry': None, 'id': '00000000000000000003_0', 'properties': {'lat': 45.872468434025315, 'lon': -119.78172435635337, 'ndvi': 0.26305702328681946, 'tile_id': 3}}, {'type': 'Feature', 'geometry': None, 'id': '00000000000000

In [8]:
import pandas as pd

cents = pd.read_csv('../data/polygons/tiles_100m2_centroids.csv')
aoi = ee.Geometry.BBox(
    cents['lon'].min(), cents['lat'].min(),
    cents['lon'].max(), cents['lat'].max()
)

years = [str(y) for y in range(2016, 2026)]

for year in years:
    start = f'{year}-01-01'
    end   = f'{year}-12-31'
    desc  = f'ndvi_100m2_{year}'

    if os.path.isfile(os.path.join(OUT_DIR, f'{desc}.csv')):
        print(f'{year}: CSV already present, skipping')
        continue

    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(aoi)            # explicit bbox — avoids asset geometry issues
          .filterDate(start, end)
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
          .select(['B4', 'B8'])
          .map(add_ndvi)
    )

    fc = collection.map(extract_image).flatten()

    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=desc,
        folder=f'ee_ndvi_100m2_{year}',
        fileFormat='CSV',
        selectors=['tile_id', 'date', 'ndvi']
    )
    task.start()
    print(f'{year}: export started → Google Drive / ee_ndvi_100m2_{year}/')

2016: export started → Google Drive / ee_ndvi_100m2_2016/
2017: export started → Google Drive / ee_ndvi_100m2_2017/
2018: export started → Google Drive / ee_ndvi_100m2_2018/
2019: export started → Google Drive / ee_ndvi_100m2_2019/
2020: export started → Google Drive / ee_ndvi_100m2_2020/
2021: export started → Google Drive / ee_ndvi_100m2_2021/
2022: export started → Google Drive / ee_ndvi_100m2_2022/
2023: export started → Google Drive / ee_ndvi_100m2_2023/
2024: export started → Google Drive / ee_ndvi_100m2_2024/
2025: export started → Google Drive / ee_ndvi_100m2_2025/


In [14]:
# Check task status
tasks = ee.batch.Task.list()
for t in tasks[:20]:
    if 'ndvi_100m2' in t.config.get('description', ''):
        print(f'{t.config["description"]:40s}  {t.state}')


ndvi_100m2_2025                           READY
ndvi_100m2_2024                           READY
ndvi_100m2_2023                           READY
ndvi_100m2_2022                           READY
ndvi_100m2_2021                           RUNNING
ndvi_100m2_2020                           RUNNING
ndvi_100m2_2019                           COMPLETED
ndvi_100m2_2018                           COMPLETED
ndvi_100m2_2017                           COMPLETED
ndvi_100m2_2016                           COMPLETED
ndvi_100m2_2025                           COMPLETED
ndvi_100m2_2024                           COMPLETED
ndvi_100m2_2023                           COMPLETED
ndvi_100m2_2022                           COMPLETED
ndvi_100m2_2021                           COMPLETED
ndvi_100m2_2020                           COMPLETED
ndvi_100m2_2019                           COMPLETED
ndvi_100m2_2018                           COMPLETED
ndvi_100m2_2017                           COMPLETED
ndvi_100m2_2016                 

## After exports complete

1. Open [Google Drive](https://drive.google.com) → download each `ee_ndvi_100m2_YYYY/` folder as a zip  
2. Place all CSVs (one per year) into `../data/ndvi/tiles_100m2/` named `ndvi_100m2_YYYY.csv`  
3. Run **02_terrain_features**, **03_soil_features**, then **04_spark_assemble**
